1. Prepare Your Dataset

Your dataset appears to be in a format similar to the CoNLL-2003 format, where each token is followed by its corresponding label. Ensure your dataset is properly formatted and split into training, validation, and test sets

Example format:

-DOCSTART- -X- O

Contact -X- _ O

www.linkedin.com -X- _ O
...

2. Install Required Libraries

Make sure you have the necessary libraries installed. You will need transformers, datasets, and torch.

In [11]:
#%pip install transformers datasets torch

3. Declare Label Names and  Load the Pre-trained Model and Tokenizer

Load the xlm-roberta-large-finetuned-conll03-english model and tokenizer using the transformers library.

In [12]:
#Declare label names
label_names = ["O", "B-AGE", "I-AGE", "B-GENDER", "I-GENDER", "B-ADDRESS", "I-ADDRESS", "B-SKILLS", "I-SKILLS", "B-EXPERIENCE", "I-EXPERIENCE", "B-EDUCATION", "I-EDUCATION", "B-CERTIFICATION", "I-CERTIFICATION",'B-Others', 'B-Role', 'I-Others', 'I-Role']

from transformers import AutoModelForTokenClassification, RobertaTokenizerFast

# Choose a smaller model
model_name = "roberta-base"  # or "distilroberta-base" or "xlm-roberta-base"

# Load the tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)

# Load the model with the correct number of labels
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),  # Set num_labels to match your dataset
    ignore_mismatched_sizes=True,  # Ignore size mismatch if necessary
)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


4. Prepare the Dataset for Training

Convert your dataset into a format that can be used by the transformers library. You can use the datasets library to load and preprocess your data.

In [13]:
from datasets import Dataset, Features, Sequence, ClassLabel, Value

# Function to load dataset
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    tokens = []
    labels = []
    current_tokens = []
    current_labels = []

    for line in lines:
        if line.strip() == "":
            if current_tokens:
                tokens.append(current_tokens)
                labels.append(current_labels)
                current_tokens = []
                current_labels = []
        else:
            parts = line.strip().split()
            current_tokens.append(parts[0])
            current_labels.append(parts[-1])

    return {"tokens": tokens, "labels": labels}

# Load your dataset
train_data = load_dataset("/content/drive/MyDrive/conlls/combined-for-train.conll")
val_data = load_dataset("/content/drive/MyDrive/conlls/combined-for-eval.conll")

# Print the structure of train_data to verify
print("Train data structure:", train_data)

# Extract unique labels
def get_label_names(dataset):
    label_names = set()
    for labels in dataset["labels"]:  # Iterate over the "labels" key
        label_names.update(labels)    # Add all labels to the set
    return sorted(list(label_names))  # Convert to a sorted list

label_names = get_label_names(train_data)
print("Label names:", label_names)
print("Number of labels:", len(label_names))
# Define features
features = Features({
    "tokens": Sequence(Value("string")),
    "labels": Sequence(ClassLabel(names=label_names))
})

# Convert to datasets.Dataset
train_dataset = Dataset.from_dict(train_data, features=features)
val_dataset = Dataset.from_dict(val_data, features=features)

Train data structure: {'tokens': [['Admin', 'Assistant', 'QUALIFICATIONS', ':', 'Graduate', 'of', 'any', 'four', '4', 'year', 'Business', 'related', 'course.', 'With', 'at', 'least', 'one', '1', 'year', 'related', 'work', 'EXPERIENCE.', 'Proficient', 'in', 'MS', 'Office.', 'Amenable', 'to', 'work', 'in', 'Pasay', 'City', '.', 'Administrative', 'Associate', 'Qualification', 'Graduate', 'of', 'any', 'Management', 'or', 'Business', 'course', 'Excellent', 'written', 'and', 'verbal', 'communication', 'SKILLS'], ['Amazon', 'Listing', 'Specialist', 'Core', 'qualifications', ':', '3-5', 'years', 'of', 'EXPERIENCE', 'with', 'Amazon', 'Seller', 'Central.', 'Expertise', 'in', 'tools', 'such', 'as', 'Helium', '10', ',', 'Jungle', 'Scout', ',', 'or', 'similar.', 'Flat-file', 'expertise', 'for', 'bulk', 'listing', 'uploads', 'and', 'updates.', 'Strong', 'ability', 'to', 'analyze', 'metrics', 'and', 'optimize', 'listings', 'based', 'on', 'data.', 'Proficiency', 'in', 'creating', 'engaging', 'and', 'a

5. Tokenize the Dataset

Tokenize the dataset using the tokenizer. Ensure that the labels are aligned with the tokenized inputs.

In [14]:
def tokenize_and_align_labels(examples):
    # Tokenize the inputs
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,          # Enable truncation
        padding="max_length",     # Pad to the maximum length
        is_split_into_words=True, # Indicate that inputs are pre-tokenized
        max_length=512,           # Set a maximum length (adjust as needed)
        return_tensors="pt",      # Return PyTorch tensors
    )

    # Align labels with tokenized inputs
    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective words
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens (e.g., [CLS], [SEP], padding) get a label of -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # New word, assign the corresponding label
                label_ids.append(label[word_idx])
            else:
                # Same word as previous token (subword tokenization), assign -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    # Add labels to the tokenized inputs
    tokenized_inputs["labels"] = labels

    # Return all necessary features, including input_ids and attention_mask
    return tokenized_inputs

# Tokenize and align labels for the train and validation datasets
train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
val_dataset = val_dataset.map(tokenize_and_align_labels, batched=True)

# Remove the "tokens" column as it's no longer needed
train_dataset = train_dataset.remove_columns(["tokens"])
val_dataset = val_dataset.remove_columns(["tokens"])

Map:   0%|          | 0/1039 [00:00<?, ? examples/s]

Map: 100%|██████████| 1039/1039 [00:00<00:00, 2071.38 examples/s]

Tokenized input IDs: [0, 43684, 6267, 16944, 2118, 36497, 14939, 4832, 26626, 9, 143, 237, 204, 76, 2090, 1330, 768, 4, 590, 23, 513, 65, 112, 76, 1330, 173, 10649, 21260, 41499, 4, 6853, 35056, 11, 6253, 1387, 4, 1918, 30743, 7, 173, 11, 11920, 857, 412, 479, 25233, 16700, 13559, 5000, 26626, 9, 143, 1753, 50, 2090, 768, 34975, 1982, 8, 14580, 4358, 14795, 10259, 104, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

6. Set Up Training Arguments

Define the training arguments using TrainingArguments.

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Reduce batch size for low memory
    per_device_eval_batch_size=8,   # Reduce batch size for low memory
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    # fp16=False,  # Disable mixed precision if not using a compatible GPU
)

7. Define the Trainer

Set up the Trainer class to handle the training loop.

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

C:\Users\Acer\AppData\Local\Temp\ipykernel_6512\1666238700.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


8. Train the Model

In [ ]:
trainer.train()

9. Evaluate the Model

In [ ]:
trainer.evaluate()

10. Save the Model

Save the fine-tuned model for future use.

In [ ]:
model.save_pretrained("./fine-tuned-model")
tokenizer.save_pretrained("./fine-tuned-tokenizer")

11. Inference

You can now use the fine-tuned model for inference on new data.

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)
results = ner_pipeline("Your input text here")
print(results)